# Word2vec: a companion notebook

**Human Language Technologies 2026-27, Lecture 11.** Claudio Gallicchio, University of Pisa.

This notebook lets you run the examples of the lecture and check the exercises of the exercise sheet. It follows Chapter 5 of Jurafsky and Martin, *Speech and Language Processing* (3rd edition draft, August 2026), sections 5.5 to 5.10.

1. The skip-gram classifier on toy embeddings
2. Choosing the noise words
3. The loss, its gradients, and one step by hand
4. A corpus without downloading: the docstrings of the Python standard library
5. Training skip-gram with negative sampling from scratch
6. Nearest neighbors
7. The size of the window
8. Analogies
9. Variability (an extra, not on the slides)
10. Checks of the exercise sheet
11. What to do next

How to use it: run the cells from top to bottom (Jupyter, VS Code or Google Colab). Only the Python standard library and `numpy` are needed, and no internet connection. Training takes about a minute in all. The exact numbers and neighbor lists of sections 4 to 9 depend on your Python version, since the corpus is its documentation: the comments in the text refer to the outputs saved in this notebook.

## 1. The skip-gram classifier on toy embeddings

The classifier of section 5.5.1: the probability that c is a real context word of the target w is the sigmoid of the dot product of their embeddings (Eq. 5.15), and the probability that it is not is the sigmoid of minus the dot product (Eq. 5.16). With many context words, the probabilities multiply (Eq. 5.17). The three toy embeddings of the lecture, with d = 3 and not learned from a corpus, plus a second noise word, *Tolstoy*.

In [1]:
import math
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

apricot = np.array([0.5, 1.0, -0.5])     # target embedding w
jam     = np.array([1.0, 0.5,  0.0])     # context embeddings c
matrix  = np.array([-0.5, 0.5, 1.0])
tolstoy = np.array([0.0, -1.0, 0.5])

for name, c in (("jam", jam), ("matrix", matrix), ("Tolstoy", tolstoy)):
    z = c @ apricot
    print(f"dot({name}, apricot) = {z:5.2f}   P(+|apricot, {name}) = {sigmoid(z):.3f}   P(-|apricot, {name}) = {sigmoid(-z):.3f}")
print("P(+|apricot, jam and matrix as the window) = %.3f" % (sigmoid(jam @ apricot) * sigmoid(matrix @ apricot)))

dot(jam, apricot) =  1.00   P(+|apricot, jam) = 0.731   P(-|apricot, jam) = 0.269
dot(matrix, apricot) = -0.25   P(+|apricot, matrix) = 0.438   P(-|apricot, matrix) = 0.562
dot(Tolstoy, apricot) = -1.25   P(+|apricot, Tolstoy) = 0.223   P(-|apricot, Tolstoy) = 0.777
P(+|apricot, jam and matrix as the window) = 0.320


## 2. Choosing the noise words

Noise words are sampled from the weighted unigram distribution P<sub>α</sub>(w), the count raised to the power α = 0.75 and normalized (Eq. 5.19): rare words get a slightly higher probability than with P(w). The two events of Eq. 5.20 first, then 10,000 samples from a small vocabulary, to see that sampling follows the weighted probabilities.

In [2]:
def p_alpha(counts, alpha=0.75):
    c = np.asarray(counts, float) ** alpha
    return c / c.sum()

print("Eq. 5.20:", p_alpha([0.99, 0.01]))
words = ["the", "of", "coffee", "cup", "aardvark"]
counts = np.array([500, 300, 150, 40, 10])
pa = p_alpha(counts)
rng = np.random.default_rng(0)
sample = rng.choice(len(words), size=10_000, p=pa)
for i, w in enumerate(words):
    print(f"{w:9s} P = {counts[i] / counts.sum():.3f}   P_alpha = {pa[i]:.3f}   sampled {np.mean(sample == i):.3f}")

Eq. 5.20: [0.969 0.031]
the       P = 0.500   P_alpha = 0.437   sampled 0.437
of        P = 0.300   P_alpha = 0.298   sampled 0.298
coffee    P = 0.150   P_alpha = 0.177   sampled 0.177
cup       P = 0.040   P_alpha = 0.066   sampled 0.064
aardvark  P = 0.010   P_alpha = 0.023   sampled 0.023


## 3. The loss, its gradients, and one step by hand

The loss of Eq. 5.21 for one positive pair and k noise words, and its gradients, Eq. 5.22 to 5.24. The gradients are checked numerically, with finite differences: a small change of one coordinate changes the loss by about the gradient times the change. Then one step of stochastic gradient descent (Eq. 5.25 to 5.27) with the toy embeddings of the lecture and η = 0.5, as on the slide *One step by hand*.

In [3]:
def sgns_loss(w, c_pos, c_negs):
    return -(np.log(sigmoid(c_pos @ w)) + sum(np.log(sigmoid(-c @ w)) for c in c_negs))

def sgns_gradients(w, c_pos, c_negs):
    g_pos = (sigmoid(c_pos @ w) - 1) * w                        # Eq. 5.22
    g_negs = [sigmoid(c @ w) * w for c in c_negs]                # Eq. 5.23
    g_w = (sigmoid(c_pos @ w) - 1) * c_pos + sum(sigmoid(c @ w) * c for c in c_negs)   # Eq. 5.24
    return g_pos, g_negs, g_w

def numerical_gradient(f, x, h=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = h
        g[i] = (f(x + e) - f(x - e)) / (2 * h)
    return g

w, c_pos, c_negs = apricot, jam, [matrix, tolstoy]
g_pos, g_negs, g_w = sgns_gradients(w, c_pos, c_negs)
print("dL/dc_pos  Eq. 5.22", g_pos, "  numerical", numerical_gradient(lambda x: sgns_loss(w, x, c_negs), c_pos))
print("dL/dc_neg1 Eq. 5.23", g_negs[0], "  numerical", numerical_gradient(lambda x: sgns_loss(w, c_pos, [x, c_negs[1]]), c_negs[0]))
print("dL/dw      Eq. 5.24", g_w, "  numerical", numerical_gradient(lambda x: sgns_loss(x, c_pos, c_negs), w))

def sgd_step(w, c_pos, c_negs, eta):
    """Eq. 5.25-5.27: all three updates use the old values"""
    g_pos, g_negs, g_w = sgns_gradients(w, c_pos, c_negs)
    return w - eta * g_w, c_pos - eta * g_pos, [c - eta * g for c, g in zip(c_negs, g_negs)]

print("\nbefore: dot products", np.array([c_pos @ w] + [c @ w for c in c_negs]), " loss %.2f" % sgns_loss(w, c_pos, c_negs))
w1, c_pos1, c_negs1 = sgd_step(w, c_pos, c_negs, eta=0.5)
print("after:  dot products", np.array([c_pos1 @ w1] + [c @ w1 for c in c_negs1]), " loss %.2f" % sgns_loss(w1, c_pos1, c_negs1))
print("new apricot", w1, "  new jam", c_pos1)

dL/dc_pos  Eq. 5.22 [-0.134 -0.269  0.134]   numerical [-0.134 -0.269  0.134]
dL/dc_neg1 Eq. 5.23 [ 0.219  0.438 -0.219]   numerical [ 0.219  0.438 -0.219]
dL/dw      Eq. 5.24 [-0.488 -0.138  0.549]   numerical [-0.488 -0.138  0.549]

before: dot products [ 1.   -0.25 -1.25]  loss 1.14
after:  dot products [ 1.524 -1.012 -1.66 ]  loss 0.68
new apricot [ 0.744  1.069 -0.775]   new jam [ 1.067  0.634 -0.067]


## 4. A corpus without downloading: the docstrings of the Python standard library

As in the notebooks of Lectures 04 and 10, the corpus is the documentation strings of more than a hundred modules of the standard library, lower-cased and tokenized, with punctuation marks as tokens. For skip-gram we keep the words that occur at least 5 times and drop the other tokens, punctuation included, before taking the windows.

In [4]:
import importlib, inspect, re, random
from collections import Counter, defaultdict

MODULES = """os re json collections itertools random statistics textwrap datetime pathlib logging argparse csv math
functools string heapq bisect difflib shutil glob fnmatch tempfile time calendar decimal fractions operator copy pprint enum
typing dataclasses abc contextlib unittest doctest pdb timeit zipfile tarfile gzip hashlib hmac secrets struct codecs io
threading queue subprocess socket email html http.client urllib.parse xml.etree.ElementTree sqlite3 pickle shelve configparser
getpass platform sys gc inspect ast tokenize keyword warnings traceback locale gettext cmd shlex optparse uuid ipaddress base64
binascii colorsys wave array weakref types numbers cmath select signal mmap asyncio concurrent.futures multiprocessing sched
graphlib zoneinfo smtplib imaplib ftplib webbrowser turtle site sysconfig venv zipapp runpy importlib pkgutil dis pickletools
filecmp linecache stat fileinput netrc plistlib mailbox mimetypes quopri unicodedata reprlib pydoc atexit errno ctypes trace
tracemalloc bdb code codeop symtable token tabnanny py_compile compileall zlib bz2 lzma""".split()

def module_text(name):
    """the docstring of a module and of its public functions and classes"""
    try:
        mod = importlib.import_module(name)
    except Exception:
        return ""
    parts = [inspect.getdoc(mod) or ""]
    for member, obj in inspect.getmembers(mod):
        if member.startswith("_") or not (inspect.isfunction(obj) or inspect.isclass(obj) or inspect.isbuiltin(obj)):
            continue
        d = inspect.getdoc(obj)
        if d and (getattr(obj, "__module__", None) or name).split(".")[0] == name.split(".")[0]:
            parts.append(d)
    return "\n".join(parts)

def tokenize(text):
    """lower-case, keep sentences of mostly words, punctuation marks as tokens"""
    text = re.sub(r"\s+", " ", text.lower())
    out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        if ">>>" in s or "\\" in s:
            continue
        toks = re.findall(r"[a-z][a-z'_-]*|[0-9]+|[^\sa-z0-9]", s)
        if len(toks) >= 3 and sum(t.isalpha() for t in toks) >= 0.7 * len(toks):
            out.extend(toks)
    return out

docs = {m: tokenize(module_text(m)) for m in MODULES}
docs = {m: t for m, t in docs.items() if len(t) >= 50}
tokens = [t for m in docs for t in docs[m]]
freq = Counter(tokens)
print(f"{len(docs)} modules, {len(tokens)} tokens, {len(freq)} types")
print("most frequent:", ", ".join(f"{w} {c}" for w, c in freq.most_common(12)))

123 modules, 60295 tokens, 4443 types
most frequent: . 3788, the 3616, a 1754, , 1747, is 1338, to 1304, of 1169, and 804, ( 755, ) 748, for 742, be 717


In [5]:
MIN_COUNT = 5
vocab = [w for w, c in freq.most_common() if w.isalpha() and c >= MIN_COUNT]
index = {w: i for i, w in enumerate(vocab)}
counts = np.array([freq[w] for w in vocab], float)
noise_p = p_alpha(counts)
seqs = [np.array([index[t] for t in docs[m] if t in index]) for m in docs]
print(f"vocabulary: {len(vocab)} words seen at least {MIN_COUNT} times; {sum(len(s) for s in seqs)} tokens kept for training")

vocabulary: 1266 words seen at least 5 times; 45182 tokens kept for training


## 5. Training skip-gram with negative sampling from scratch

The algorithm of section 5.5.2. Every (target, context) pair within ±L words is a positive example; for each, k = 5 noise words are drawn from P<sub>α</sub>. W and C start as small random matrices, and stochastic gradient descent applies Eq. 5.25 to 5.27, here to 1,024 pairs at a time, whose updates are computed with the same old values and applied together, so that numpy can do the work. A noise word that happens to be the target word is drawn again. The average loss per pair goes down epoch after epoch. Each word is then represented by w<sub>i</sub> + c<sub>i</sub>.

In [6]:
def positive_pairs(seqs, L):
    """all (target, context) pairs within L words, in each document"""
    out = []
    for s in seqs:
        for off in range(1, L + 1):
            if len(s) > off:
                out.append(np.stack([s[:-off], s[off:]], 1)); out.append(np.stack([s[off:], s[:-off]], 1))
    return np.concatenate(out)

def train_sgns(seqs, L=2, d=50, k=5, epochs=5, eta=0.025, batch=1024, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    P = positive_pairs(seqs, L)
    V = len(vocab)
    W = (rng.random((V, d)) - 0.5) / d       # target embeddings
    C = (rng.random((V, d)) - 0.5) / d       # context embeddings
    for ep in range(epochs):
        P = P[rng.permutation(len(P))]
        N = rng.choice(V, size=(len(P), k), p=noise_p)       # k noise words for each positive pair
        clash = N == P[:, :1]                                 # a noise word must not be the target word w
        while clash.any():
            N[clash] = rng.choice(V, size=int(clash.sum()), p=noise_p); clash = N == P[:, :1]
        total = 0.0
        for b in range(0, len(P), batch):
            wi, ci, ni = P[b:b + batch, 0], P[b:b + batch, 1], N[b:b + batch]
            w, cp, cn = W[wi], C[ci], C[ni]
            sp = sigmoid(np.einsum("bd,bd->b", cp, w))            # P(+|w, c_pos)
            sn = sigmoid(np.einsum("bkd,bd->bk", cn, w))          # P(+|w, c_neg), to be pushed down
            total -= np.log(sp).sum() + np.log(1 - sn).sum()      # Eq. 5.21
            g_cp = (sp - 1)[:, None] * w                          # Eq. 5.22
            g_cn = sn[:, :, None] * w[:, None, :]                 # Eq. 5.23
            g_w = (sp - 1)[:, None] * cp + np.einsum("bk,bkd->bd", sn, cn)   # Eq. 5.24
            np.add.at(C, ci, -eta * g_cp)
            np.add.at(C, ni.ravel(), -eta * g_cn.reshape(-1, d))
            np.add.at(W, wi, -eta * g_w)
        if verbose: print(f"window ±{L}, epoch {ep + 1}: average loss per positive pair {total / len(P):.3f}")
    return W, C, len(P)

W2, C2, n_pairs = train_sgns(seqs, L=2)
print(f"{n_pairs} positive pairs per epoch, each with 5 noise words")
E2 = W2 + C2

window ±2, epoch 1: average loss per positive pair 3.121


window ±2, epoch 2: average loss per positive pair 2.607


window ±2, epoch 3: average loss per positive pair 2.547


window ±2, epoch 4: average loss per positive pair 2.513


window ±2, epoch 5: average loss per positive pair 2.472
179990 positive pairs per epoch, each with 5 noise words


## 6. Nearest neighbors

The nearest neighbors by cosine of some of the words of the notebook of Lecture 10, now with dense embeddings of 50 dimensions instead of rows of a 2000 × 2000 count matrix. With 45,000 training tokens the lists are noisy, but many neighbors make sense.

In [7]:
def neighbors(E, word, k=8):
    v = E[index[word]]
    norms = np.linalg.norm(E, axis=1); norms[norms == 0] = 1
    cos = E @ v / (norms * np.linalg.norm(v))
    return [(vocab[j], round(float(cos[j]), 2)) for j in np.argsort(-cos) if vocab[j] != word][:k]

for w in ("file", "string", "error", "true", "list", "directory", "thread", "socket"):
    print(f"{w:10s}", neighbors(E2, w))

file       [('source', 0.73), ('compiled', 0.73), ('opened', 0.67), ('open', 0.65), ('executable', 0.63), ('binary', 0.63), ('descriptor', 0.6), ('plist', 0.6)]
string     [('containing', 0.79), ('identifying', 0.73), ('ascii', 0.72), ('single', 0.69), ('query', 0.69), ('byte', 0.67), ('doc', 0.67), ('document', 0.67)]
error      [('handling', 0.8), ('encoding', 0.68), ('occurs', 0.62), ('which', 0.62), ('raised', 0.62), ('mailbox', 0.61), ('created', 0.61), ('binascii', 0.6)]
true       [('false', 0.82), ('legacy', 0.7), ('otherwise', 0.69), ('else', 0.68), ('policy', 0.67), ('capture', 0.67), ('iff', 0.63), ('return', 0.63)]
list       [('lines', 0.72), ('tuples', 0.71), ('hexadecimal', 0.71), ('elements', 0.67), ('host', 0.65), ('sequence', 0.65), ('int', 0.64), ('giving', 0.64)]
directory  [('package', 0.75), ('tree', 0.73), ('pickle', 0.58), ('cache', 0.56), ('node', 0.56), ('killed', 0.55), ('signal', 0.54), ('found', 0.54)]
thread     [('library', 0.79), ('main', 0.65), ('hierar

Some lists are good: *file* is closest to *source*, *compiled*, *opened*, *open*, *executable*, *binary*; *error* has *handling*, *occurs* and *raised* among its neighbors, *list* has *lines*, *tuples*, *elements* and *sequence*. *true* is closest to *false*: antonyms occur in the same contexts, and distributional similarity does not tell them apart. Other lists, like that of *socket*, are mostly noise: 45,000 tokens are very little for word2vec, whose pretrained vectors were trained on about 100 billion words of news.

## 7. The size of the window

Section 5.7: short windows tend to give similar words with the same part of speech, long windows words that are topically related. The same training with a window of ±8, for three epochs only: there are almost four times as many pairs.

In [8]:
W8, C8, n8 = train_sgns(seqs, L=8, epochs=3)
E8 = W8 + C8
print(f"{n8} positive pairs per epoch, {n8 / n_pairs:.1f} times those of the window ±2\n")
for w in ("file", "error", "directory", "thread", "socket"):
    print(f"{w:10s} ±2", [x for x, _ in neighbors(E2, w, 6)])
    print(f"{'':10s} ±8", [x for x, _ in neighbors(E8, w, 6)])

window ±8, epoch 1: average loss per positive pair 2.766


window ±8, epoch 2: average loss per positive pair 2.588


window ±8, epoch 3: average loss per positive pair 2.546
714056 positive pairs per epoch, 4.0 times those of the window ±2

file       ±2 ['source', 'compiled', 'opened', 'open', 'executable', 'binary']
           ±8 ['wave', 'opened', 'searched', 'executable', 'plist', 'binary']
error      ±2 ['handling', 'encoding', 'occurs', 'which', 'raised', 'mailbox']
           ±8 ['msg', 'raised', 'handling', 's', 'occurs', 'binascii']
directory  ±2 ['package', 'tree', 'pickle', 'cache', 'node', 'killed']
           ±8 ['top', 'relative', 'working', 'package', 'sys', 'destination']
thread     ±2 ['library', 'main', 'hierarchy', 'execution', 'signature', 'module']
           ±8 ['loop', 'signals', 'event', 'control', 'release', 'threads']
socket     ±2 ['browser', 'launcher', 'required', 'creating', 'database', 'paramspec']
           ±8 ['do', 'support', 'o', 'connected', 'browser', 'adds']


With ±8 the neighbors of *directory* (*top*, *relative*, *working*, *destination*) and of *thread* (*loop*, *signals*, *event*, *threads*) are words of the same topic; with ±2 the lists are less clearly topical. On such a small corpus the difference between the two windows is not sharp, and both lists mix parts of speech.

## 8. Analogies

The parallelogram model (Eq. 5.28): for a : b :: a* : ?, the word nearest to b − a + a*, excluding the three input words. With the cosine as the measure of nearness, as is common; the book's Eq. 5.28 uses a distance. Here a few analogies on our embeddings; the toy vectors of exercise 10 of the sheet are in section 10.

In [9]:
def analogy(E, a, b, a_star, k=3, exclude=True):
    t = E[index[b]] - E[index[a]] + E[index[a_star]]
    norms = np.linalg.norm(E, axis=1); norms[norms == 0] = 1
    cos = E @ t / (norms * np.linalg.norm(t))
    out = [vocab[j] for j in np.argsort(-cos) if not (exclude and vocab[j] in (a, b, a_star))]
    return out[:k]

tests = [("key", "keys", "value"), ("file", "files", "string"), ("line", "lines", "file"), ("read", "write", "input"), ("byte", "bytes", "character")]
for a, b, a_star in tests:
    print(f"{a} : {b} :: {a_star} : ?   ", analogy(E2, a, b, a_star), "   without excluding the inputs:", analogy(E2, a, b, a_star, exclude=False))

key : keys :: value : ?    ['values', 'options', 'field']    without excluding the inputs: ['values', 'value', 'options']
file : files :: string : ?    ['imap', 'unicode', 'directories']    without excluding the inputs: ['imap', 'unicode', 'directories']
line : lines :: file : ?    ['compressed', 'descriptor', 'parse']    without excluding the inputs: ['compressed', 'descriptor', 'file']
read : write :: input : ?    ['constructor', 'string', 'configuration']    without excluding the inputs: ['constructor', 'string', 'configuration']
byte : bytes :: character : ?    ['aninteger', 'array', 'create']    without excluding the inputs: ['bytes', 'character', 'aninteger']


Only *key : keys :: value : ?* gives the expected answer, *values*. With 45,000 tokens the parallelogram fails: the analogies of the book come from embeddings trained on billions of words. Note also the caveat of the lecture: without excluding the inputs, *byte : bytes :: character : ?* returns *bytes*, one of the input words, before anything else.

## 9. Variability (an extra, not on the slides)

An extra, not on the slides (section 5.9 of the book): because of the random initialization and the random negative sampling, word2vec can give different results from the same data. The same training as in section 5, with another seed: how many of the 8 nearest neighbors stay the same?

In [10]:
W2b, C2b, _ = train_sgns(seqs, L=2, seed=1, verbose=False)
E2b = W2b + C2b
for w in ("file", "string", "error", "true", "list", "directory", "thread", "socket"):
    a = {x for x, _ in neighbors(E2, w)}; b = {x for x, _ in neighbors(E2b, w)}
    print(f"{w:10s} {len(a & b)} of 8 in common:", sorted(a & b))

file       7 of 8 in common: ['binary', 'compiled', 'descriptor', 'open', 'opened', 'plist', 'source']
string     7 of 8 in common: ['ascii', 'byte', 'containing', 'document', 'identifying', 'query', 'single']
error      7 of 8 in common: ['binascii', 'created', 'encoding', 'handling', 'occurs', 'raised', 'which']
true       6 of 8 in common: ['capture', 'else', 'false', 'legacy', 'otherwise', 'policy']
list       5 of 8 in common: ['giving', 'hexadecimal', 'host', 'lines', 'tuples']
directory  4 of 8 in common: ['node', 'package', 'signal', 'tree']
thread     5 of 8 in common: ['hierarchy', 'library', 'main', 'module', 'signature']
socket     6 of 8 in common: ['browser', 'creating', 'database', 'factory', 'launcher', 'required']


Between 4 and 7 of the 8 neighbors stay the same: the lists of *file*, *string* and *error* are the most stable, that of *directory* the least. This is why, when embeddings are used to study word associations, the book recommends training several embeddings on bootstrap samples of the documents and averaging the results.

## 10. Checks of the exercise sheet

The numerical exercises of the sheet, recomputed: exercises 1 to 5, 7 and 9 to 12 (exercise 6 is a derivation, checked numerically in section 3; exercise 8 needs no numbers). Do them by hand first.

In [11]:
# exercise 1: weights and parameters
print("exercise 1   weights: %d sparse, %d dense; skip-gram parameters 2 x 50,000 x 300 = %d" % (50_000, 300, 2 * 50_000 * 300))

# exercise 2: positive pairs of the sentence, window 2
sentence = "the quick brown fox jumps over the lazy dog".split()
pairs2 = [(sentence[i], sentence[j]) for i in range(len(sentence)) for j in range(max(0, i - 2), min(len(sentence), i + 3)) if j != i]
print("exercise 2   pairs for fox:", [p for p in pairs2 if p[0] == "fox"], " all positive pairs:", len(pairs2), " negative pairs with k = 3:", 3 * len(pairs2))

# exercise 3: the classifier by hand
coffee, cup, drink, tax, seven = (np.array(v, float) for v in ([1, 2, 0], [1, 1, 1], [0, 1, -1], [-1, 0, 1], [0, -1, 1]))
for name, c in (("cup", cup), ("drink", drink), ("tax", tax), ("seven", seven)):
    print(f"exercise 3   dot(coffee, {name}) = {c @ coffee:4.1f}   P(+) = {sigmoid(c @ coffee):.3f}   P(-) = {sigmoid(-c @ coffee):.3f}")
p_window = sigmoid(cup @ coffee) * sigmoid(drink @ coffee)
print(f"             P(+|coffee, cup and drink) = {p_window:.3f}, log = {math.log(p_window):.3f}")

# exercise 4: the noise words
print("exercise 4   P =", counts_ex4 := np.array([500, 300, 150, 40, 10]) / 1000, " counts^0.75 =", (np.array([500, 300, 150, 40, 10]) ** 0.75).round(2), " P_alpha =", p_alpha([500, 300, 150, 40, 10]).round(3))
print("             expected out of 1,000 noise words: the", 1000 * 0.5, "and", round(1000 * p_alpha([500, 300, 150, 40, 10])[0], 1), "; aardvark", 1000 * 0.01, "and", round(1000 * p_alpha([500, 300, 150, 40, 10])[4], 1))

# exercise 5: the loss
print("exercise 5   L = %.3f   terms:" % sgns_loss(coffee, cup, [tax, seven]), [round(float(-np.log(sigmoid(x))), 3) for x in (cup @ coffee, -(tax @ coffee), -(seven @ coffee))], " all dot products 0: %.3f" % (3 * math.log(2)))

exercise 1   weights: 50000 sparse, 300 dense; skip-gram parameters 2 x 50,000 x 300 = 30000000
exercise 2   pairs for fox: [('fox', 'quick'), ('fox', 'brown'), ('fox', 'jumps'), ('fox', 'over')]  all positive pairs: 30  negative pairs with k = 3: 90
exercise 3   dot(coffee, cup) =  3.0   P(+) = 0.953   P(-) = 0.047
exercise 3   dot(coffee, drink) =  2.0   P(+) = 0.881   P(-) = 0.119
exercise 3   dot(coffee, tax) = -1.0   P(+) = 0.269   P(-) = 0.731
exercise 3   dot(coffee, seven) = -2.0   P(+) = 0.119   P(-) = 0.881
             P(+|coffee, cup and drink) = 0.839, log = -0.176
exercise 4   P = [0.5  0.3  0.15 0.04 0.01]  counts^0.75 = [105.74  72.08  42.86  15.91   5.62]  P_alpha = [0.437 0.298 0.177 0.066 0.023]
             expected out of 1,000 noise words: the 500.0 and 436.5 ; aardvark 10.0 and 23.2
exercise 5   L = 0.489   terms: [0.049, 0.313, 0.127]  all dot products 0: 2.079


In [12]:
# exercise 7: one step of SGD
w7, cpos7, cneg7 = np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0])
g_pos, g_negs, g_w = sgns_gradients(w7, cpos7, [cneg7])
print("exercise 7   before: dots", cpos7 @ w7, cneg7 @ w7, " loss %.3f" % sgns_loss(w7, cpos7, [cneg7]))
print("             gradients", g_pos, g_negs[0], g_w)
for eta in (0.5, 1.0):
    w_, cp_, cn_ = sgd_step(w7, cpos7, [cneg7], eta)
    print(f"             eta = {eta}: w = {w_}, c_pos = {cp_}, c_neg = {cn_[0]}; dots {cp_ @ w_:.3f}, {cn_[0] @ w_:.3f}; loss {sgns_loss(w_, cp_, cn_):.3f}")

# exercise 9: fasttext n-grams
def char_ngrams(word, n):
    s = "<" + word + ">"
    return [s[i:i + n] for i in range(len(s) - n + 1)]
print("exercise 9  ", char_ngrams("where", 3), char_ngrams("cats", 3), char_ngrams("catlike", 3), char_ngrams("cats", 4))
print("             shared by catlike and cats:", sorted(set(char_ngrams("catlike", 3)) & set(char_ngrams("cats", 3))))

# exercise 10: the parallelogram with the Euclidean distance (Eq. 5.28)
toy = {w: np.array(v, float) for w, v in {"man": [1, 1], "woman": [1, 3], "king": [4, 1], "queen": [4, 3.2], "princess": [3, 3.5],
                                          "cherry": [3, 3], "red": [2, 3], "potato": [6, 0], "potatoes": [6.5, 0.5], "brown": [5, 1.5]}.items()}
def parallelogram(a, b, a_star, candidates):
    t = toy[b] - toy[a] + toy[a_star]
    return t, sorted((round(float(np.linalg.norm(toy[x] - t)), 3), x) for x in candidates)
print("exercise 10 ", parallelogram("man", "woman", "king", ["man", "woman", "king", "queen", "princess"]))
print("            ", parallelogram("cherry", "red", "potato", ["cherry", "red", "potato", "potatoes", "brown"]))

# exercise 11: a bias score
he, she = np.array([1.0, 0.0]), np.array([0.0, 1.0])
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
for name, v in (("nurse", [0.2, 0.9]), ("engineer", [0.9, 0.3]), ("teacher", [0.6, 0.6])):
    v = np.array(v); print(f"exercise 11  {name:9s} cos(she) {cos(v, she):.3f}  cos(he) {cos(v, he):.3f}  difference {cos(v, she) - cos(v, he):+.3f}")

# exercise 12: Spearman rank correlation
human = np.array([9.1, 5.8, 4.5, 6.0, 1.2]); model = np.array([0.71, 0.55, 0.62, 0.48, 0.10])
rank = lambda x: np.argsort(np.argsort(-x)) + 1
d = rank(human) - rank(model)
print("exercise 12  human ranks", rank(human), " model ranks", rank(model), " rho = %.2f" % (1 - 6 * (d ** 2).sum() / (5 * (5 ** 2 - 1))))

exercise 7   before: dots 0.0 1.0  loss 2.006
             gradients [-0.5 -0. ] [0.731 0.   ] [0.731 0.231]
             eta = 0.5: w = [ 0.634 -0.116], c_pos = [0.25 1.  ], c_neg = [0.634 1.   ]; dots 0.043, 0.287; loss 1.519
             eta = 1.0: w = [ 0.269 -0.231], c_pos = [0.5 1. ], c_neg = [0.269 1.   ]; dots -0.097, -0.159; loss 1.360
exercise 9   ['<wh', 'whe', 'her', 'ere', 're>'] ['<ca', 'cat', 'ats', 'ts>'] ['<ca', 'cat', 'atl', 'tli', 'lik', 'ike', 'ke>'] ['<cat', 'cats', 'ats>']
             shared by catlike and cats: ['<ca', 'cat']
exercise 10  (array([4., 3.]), [(0.2, 'queen'), (1.118, 'princess'), (2.0, 'king'), (3.0, 'woman'), (3.606, 'man')])
             (array([5., 0.]), [(1.0, 'potato'), (1.5, 'brown'), (1.581, 'potatoes'), (3.606, 'cherry'), (4.243, 'red')])
exercise 11  nurse     cos(she) 0.976  cos(he) 0.217  difference +0.759
exercise 11  engineer  cos(she) 0.316  cos(he) 0.949  difference -0.632
exercise 11  teacher   cos(she) 0.707  cos(he) 0.707  differe

## 11. What to do next

- The exercise sheet of this lecture: section 10 recomputes its numerical exercises; `sgns_loss`, `sgns_gradients`, `sgd_step` and `p_alpha` work on any example of your own.
- Train with other settings: more epochs, d = 100, k = 10, a minimum count of 3 instead of 5, the vectors W alone instead of W + C. Which neighbor lists improve?
- Replace the docstrings with a text of your choice, a public-domain novel for example: with a much larger corpus the neighbors, and the analogies of section 8, should get better.
- Compare the neighbors of section 6 with those of the PPMI matrix in the notebook of Lecture 10, on the same corpus.